# 第 15 章:进阶 —— 知识蒸馏与 MoE

这是全书的最后一章。我们学两个进阶主题:

1. **知识蒸馏(Knowledge Distillation)**:用大模型(teacher)教小模型(student)
2. **混合专家(Mixture of Experts, MoE)**:多个子网络各司其职,按需激活

这两个技术在 minimind 中都有完整实现,也是现代 LLM 的重要优化方向。

## 15.1 知识蒸馏:为什么需要

训练好的模型面临一个矛盾:

- **大模型**能力强,但推理慢、显存大
- **小模型**快且轻,但能力弱

**知识蒸馏**的思路:让小模型「模仿」大模型的输出分布,而不仅仅是模仿 ground-truth 标签。

读 `train_distillation.py`,minimind 的实现:

- **Teacher**:MoE 模型(198M-A64M),能力强但部署难
- **Student**:Dense 模型(64M),轻量好部署
- **目标**:让 student 的输出分布尽可能接近 teacher

## 15.2 蒸馏 Loss:温度与暗知识

蒸馏的核心 loss 是 **KL 散度**:

$$\mathcal{L}_{\text{KL}} = T^2 \cdot D_{\text{KL}}\left(\text{softmax}(z_s / T) \| \text{softmax}(z_t / T)\right)$$

- $z_s$:student 的 logits
- $z_t$:teacher 的 logits
- $T$:**温度**(temperature)

读 `distillation_loss`(`train_distillation.py:~25-36 (@67f114a)`):

```python
def distillation_loss(student_logits, teacher_logits, temperature):
    # 温度缩放:soften 分布
    # T 大 → 分布更平坦 → 暴露更多「暗知识」
    student_soft = F.log_softmax(student_logits / temperature, dim=-1)
    teacher_soft = F.softmax(teacher_logits / temperature, dim=-1)
    # KL 散度
    loss = F.kl_div(student_soft, teacher_soft, reduction='batchmean')
    # T² 缩放:补偿梯度幅度(因为 /T 压缩了 logits → 梯度变小)
    return loss * (temperature ** 2)
```

### 温度 T 的作用

温度控制 softmax 分布的「软度」:

- **T=1**:标准 softmax —— 几乎所有概率集中在 top-1(一个 token 概率 ~0.9,其余 ~0.01)
- **T=5**:更平坦的分布 —— top-1 概率降到 ~0.3,但其他 token 的**相对关系**暴露出来

> **暗知识(Dark Knowledge)**:teacher 对非 top-1 token 的相对打分。比如 teacher 认为 token A 比 token B 更合理(即使两者都不是最优),这个信息在 T=1 时几乎不可见,但在 T=5 时变成了有意义的梯度信号。

## 15.3 组合 Loss

蒸馏不只看 teacher 的分布,也要看 ground-truth:

$$\mathcal{L} = \alpha \cdot \mathcal{L}_{\text{CE}} + (1 - \alpha) \cdot \mathcal{L}_{\text{KL}}$$

读组合 loss(`train_distillation.py:~92-93 (@67f114a)`):

```python
# α=0.5: 平衡硬标签(ground-truth)和软标签(teacher)
ce_loss = F.cross_entropy(student_logits, labels, ignore_index=-100)
kl_loss = distillation_loss(student_logits, teacher_logits, temperature=T)
total_loss = alpha * ce_loss + (1 - alpha) * kl_loss
```

| 参数 | 值 | 说明 |
|---|---|---|
| α | 0.5 | CE 和 KL 各占一半 |
| T | 1.5 | 适度软化(不用太大) |
| lr | 5e-6 | 极低(只是微调) |
| epochs | 6 | 多轮蒸馏 |

> 默认配置:teacher=MoE, student=dense。把 MoE 的能力压缩进 dense,方便部署。

## 15.4 MoE:混合专家架构

**Mixture of Experts(MoE)**:不用一个大 FFN,而是用 N 个小 FFN(专家),每个 token 只激活其中 1 个。

minimind 的 MoE 配置(`MiniMindConfig`):
- `num_experts = 4`:4 个专家 FFN
- `num_experts_per_tok = 1`:每个 token 只激活 1 个(top-1 路由)
- Dense 参数量:64M;MoE 参数量:198M;**激活参数量:~64M**(只激活 1/4 的 FFN)

## 15.5 MoE 路由:Gate + Top-1

读 `MOEFeedForward`(`model_minimind.py:~153-181 (@67f114a)`):

```python
class MOEFeedForward(nn.Module):
    def __init__(self, config):
        self.experts = nn.ModuleList([
            FeedForward(config) for _ in range(config.num_experts)
        ])  # 4 个独立的 SwiGLU FFN
        self.gate = nn.Linear(config.hidden_size, config.num_experts, bias=False)
        # 路由器:hidden → 4

    def forward(self, x):
        # x Shape: (batch, seq_len, hidden_size)
        router_logits = self.gate(x)
        # Shape: (batch, seq_len, num_experts=4)
        routing_weights = F.softmax(router_logits, dim=-1)
        # Shape: (batch, seq_len, 4) — 每个 token 对 4 个专家的偏好

        # Top-1 路由:选概率最大的专家
        expert_idx = routing_weights.argmax(dim=-1)
        # Shape: (batch, seq_len) — 每个 token 选一个专家

        # 派发:把 token 发给选中的专家
        for i, expert in enumerate(self.experts):
            mask = (expert_idx == i)  # 哪些 token 选了专家 i
            if mask.any():
                x[mask] = expert(x[mask])  # 只处理分给这个专家的 token

        return x
```

## 15.6 负载均衡:Aux Loss

MoE 有一个致命问题:**路由崩塌(Routing Collapse)**。如果一开始某个专家表现稍好,路由器会越来越偏向它,其他专家得不到训练,最终所有 token 都走同一个专家 —— 等于退化成 dense 模型。

**解决方案**:auxiliary loss(辅助损失),鼓励负载均匀:

$$\mathcal{L}_{\text{aux}} = \sum_{i=1}^{N} f_i \cdot P_i$$

- $f_i$:专家 $i$ **实际收到**的 token 比例(负载频率)
- $P_i$:路由器给专家 $i$ 的**平均概率**

读 `MOEFeedForward`(~176-178 (@67f114a)):

```python
# aux_loss = load · scores.mean
# load: 每个专家实际处理的 token 比例
# scores.mean: 路由器对每个专家的平均打分
aux_loss = (tokens_per_expert / total_tokens) * routing_weights.mean(dim=[0,1])
# 当负载均匀时 aux_loss 最小;当所有 token 走同一个专家时 aux_loss 最大
```

`router_aux_loss_coef = 5e-4`(配置第 ~50 行 (@67f114a))—— aux_loss 只占总 loss 的很小一部分,足以引导均衡,但不干扰主任务。

> **梯度技巧**(第 ~174 行 (@67f114a)):未选中的专家也有梯度流(通过 routing_weights 的 softmax),确保它们能更新。

## 15.7 Dense vs MoE 对比

| 维度 | Dense(64M) | MoE(198M-A64M) |
|---|---|---|
| 总参数量 | 64M | 198M |
| 每 token 激活参数 | 64M | ~64M(只激活 1/4 FFN) |
| 推理速度 | 基准 | 略慢(路由开销) |
| 能力 | 基准 | 更强(更多总参数) |
| 显存 | 基准 | ~3x(需加载所有专家) |
| 部署难度 | 简单 | 更复杂 |

> MoE 的核心价值:**在不增加推理计算量的前提下提升模型能力**。代价是显存增大(所有专家都要加载到内存,即使每次只用一个)。

&nbsp;

---

## Summary and takeaways

恭喜!你完成了全部 15 章的学习!

**本章回顾**:

- **知识蒸馏**:用 teacher 的软标签(T° 缩放后的分布)训练 student;暗知识 = 非最优 token 的相对打分;`α·CE + (1-α)·T²·KL`
- **MoE**:多专家 FFN + 门控路由;top-1 激活;aux_loss 防止路由崩塌;总参数大但激活参数不变

**全书回顾** —— 你现在理解了:

| 部分 | 章节 | 你学会了 |
|---|---|---|
| 模型半 | ch1-7 | 从 tokenizer 到 attention 到 generate,手写一个完整 Transformer |
| 训练半 | ch8-11 | 预训练、SFT(loss masking)、LoRA、推理部署 |
| 对齐半 | ch12-15 | DPO、PPO/GRPO/CISPO 统一框架、Agent-RL、蒸馏与 MoE |

> **核心认知**:LLM 不是黑盒。它是分词器 + 嵌入 + 多层 Attention/FFN + 采样,通过预训练获得语言能力,SFT 学会对话,RL 对齐变「有用」。每一层你都能从零写出、理解、修改。

> 「亲手用乐高搭一架飞机,远比坐头等舱更令人兴奋。」

- 精简复习版见 [`./advanced.ipynb`](./advanced.ipynb)
- 本章习题与解答见 [`./exercise-solutions.ipynb`](./exercise-solutions.ipynb)

← [教程总览](../../README.md)